# Benamou--Brenier geodesic between two shapes

This notebook generates `fig:dynamic-benamou-brenier-geodesic`.  It samples the cat and two-disks silhouettes, computes a discrete quadratic OT plan with POT, and displays the McCann interpolation
$$
    Z_t=(1-t)X+tY, \qquad V_t=Y-X.
$$
The first panel shows kernel-density contours along the geodesic; the second overlays midpoint particles with small velocity arrows.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME='dynamic-benamou-brenier-geodesic'; out=figure_dir(NAME)
def sample_mask(path,n,seed):
    im=Image.open(path).convert('L').resize((130,130))
    arr=np.asarray(im)/255.0
    mask=arr<0.72 if arr.mean()>0.5 else arr>0.28
    yy,xx=np.where(mask)
    rr=np.random.default_rng(seed)
    idx=rr.choice(len(xx),size=n,replace=len(xx)<n)
    pts=np.c_[xx[idx], -yy[idx]].astype(float)
    pts-=pts.mean(0); pts/=max(np.linalg.norm(pts,axis=1).max(),1e-12)
    return pts
x=sample_mask(ROOT/'notebooks-figures/assets/cat.png',260,12)+np.array([-0.85,0])
y=sample_mask(ROOT/'notebooks-figures/assets/twodisks.png',260,13)+np.array([0.85,0])
a=np.ones(len(x))/len(x); b=np.ones(len(y))/len(y); C=ot.dist(x,y)
P=ot.emd(a,b,C); match=P.argmax(axis=1); ym=y[match]; vel=ym-x
allp=np.vstack([x,y]); xlim,ylim=padded_limits(allp,pad=0.18)
gx=np.linspace(xlim[0],xlim[1],90); gy=np.linspace(ylim[0],ylim[1],78); X,Y=np.meshgrid(gx,gy); G=np.stack([X,Y],axis=-1)
def kde(z,sigma=0.105):
    D=((G[:,:,None,:]-z[None,None,:,:])**2).sum(axis=-1)
    den=np.exp(-D/(2*sigma**2)).mean(axis=-1); return den/max(den.max(),1e-12)
fig,axes=plt.subplots(1,5,figsize=(5.7,1.30),gridspec_kw={'wspace':0.02})
for ax,t in zip(axes,[0,.25,.5,.75,1]):
    z=(1-t)*x+t*ym; color=interp_color(t); den=kde(z)
    ax.contourf(X,Y,den,levels=np.linspace(.10,1,9),colors=[color],alpha=0.12)
    ax.contour(X,Y,den,levels=[.22,.40,.60,.80],colors=[color],linewidths=0.48,alpha=0.74)
    ax.scatter(z[::4,0],z[::4,1],s=DIRAC_MARKER_SIZE*.42,color=color,marker='o',edgecolor='none',linewidth=0,alpha=0.72)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'density-sequence.pdf',pad_inches=0.04); plt.close(fig)
z=.5*x+.5*ym; den=kde(z); fig,ax=plt.subplots(figsize=(2.6,1.85))
ax.contourf(X,Y,den,levels=np.linspace(.10,1,9),colors=[VIOLET],alpha=0.13)
ax.contour(X,Y,den,levels=[.22,.40,.60,.80],colors=[VIOLET],linewidths=0.52,alpha=0.76)
sub=np.arange(0,len(z),10); ax.quiver(z[sub,0],z[sub,1],vel[sub,0],vel[sub,1],color=GRAY,angles='xy',scale_units='xy',scale=13,width=0.0048,alpha=0.70)
ax.scatter(z[sub,0],z[sub,1],s=DIRAC_MARKER_SIZE*.58,color=VIOLET,marker='o',edgecolor='none',linewidth=0,zorder=3)
ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'midpoint-velocities.pdf',pad_inches=0.045); plt.close(fig)